# Çizim duygu sınıflandırma — Colab

Bu notebook sadece `src/` fonksiyonlarını çağırıp sonuç gösterir. Kod değişikliği yerelde repoda yapılır,
burada `git pull` ile çekilir. Pull'dan sonra değişen kodun yüklenmesi için **Runtime > Restart session** yapıp
hücreleri baştan çalıştır.

Eski keşif ve deneme hücreleri `grad_proj.ipynb`'de (arşiv).

In [ ]:
from google.colab import drive, userdata
drive.mount("/content/drive", force_remount=True)

import importlib, os, subprocess, sys
REPO = "/content/child-drawing-analysis"
URL = "github.com/nia1da/child-drawing-analysis.git"
# Repo gizliyse Colab Secrets'taki GITHUB_TOKEN kullanılır (sadece bu repo, Contents: Read-only); public ise gerekmez
try:
    yetkili = f"https://x-access-token:{userdata.get('GITHUB_TOKEN')}@{URL}"
except Exception:
    yetkili = f"https://{URL}"

def git(*args):
    # Hata mesajında token görünmesin diye komut yazdırılmaz
    r = subprocess.run(["git", *args], capture_output=True, text=True)
    if r.returncode != 0:
        raise RuntimeError("git hatası: " + r.stderr.replace(yetkili, f"https://{URL}"))

if os.path.isdir(f"{REPO}/.git"):
    git("-C", REPO, "pull", "-q", yetkili, "main")
else:
    git("clone", "-q", yetkili, REPO)
    git("-C", REPO, "remote", "set-url", "origin", f"https://{URL}")  # token .git/config'te kalmasın
print(subprocess.run(["git", "-C", REPO, "log", "--oneline", "-1"], capture_output=True, text=True).stdout)

if REPO not in sys.path:
    sys.path.insert(0, REPO)
importlib.invalidate_caches()  # önceki başarısız denemeden kalan "klasör yok" önbelleğini temizler

In [ ]:
import pandas as pd
from src.config import ENV, PATHS
from src.data import prepare_data, load_metadata
from src.training import train, device
from src.evaluation import evaluate
print("Ortam:", ENV, "| cihaz:", device)

prepare_data()  # KIDO zip'ini Drive'dan /content/kido altına açar; zip Drive'da yoksa önce indirip oraya kaydeder
df = load_metadata()

# deneyler.csv eski 4 sütunlu düzendeyse 12 sütuna çevirir (orijinal deneyler_eski.csv olarak kalır);
# zaten yeni düzendeyse bir şey yapmaz
from src.migrate_experiments import migrate
if os.path.exists(f"{PATHS['output']}/deneyler.csv"):
    migrate()
print(df.bolum.value_counts())

## Test değerlendirmesi

Test seti sadece seçilmiş modellerin son kıyaslaması için kullanılır; model ve hiperparametre seçimi val F1 ile yapılır.

In [ ]:
DENEYLER = [
    "convnext_tiny_448px_lr0.0001_ls0.1_s42",
    "resnet50_448px_lr0.0001_ls0.1_s42",
]
test = {d: evaluate(df, d) for d in DENEYLER}
pd.DataFrame({d: r["overall"] for d, r in test.items()}).T

## Alt grup güven aralıkları

Öğrenci bazlı bootstrap (2000 tekrar) ile %95 güven aralıkları. Kayıtlı tahmin dosyalarından hesaplanır,
model yeniden çalıştırılmaz. Fark aralığı 0'ı içermiyorsa `anlamli = True`.

In [ ]:
from src.stats import confidence_intervals, load_predictions
for d in DENEYLER:
    gruplar, farklar = confidence_intervals(load_predictions(d))
    print(f"== {d} ==")
    display(gruplar, farklar)

## Güven bandı

Eşik val tahminlerinden belirlenir: "emin" denen tahminlerin val doğruluğu en az `HEDEF` olacak şekilde en düşük güven
(güven = |p(üzgün) − 0.5| × 2). Eşik `{deney}_esik.json` olarak modelin yanına kaydedilir; test tablosu sadece raporlama içindir.

In [ ]:
from src.analysis import compare, confidence_band, error_direction, load_threshold, save_threshold
from src.stats import load_predictions

HEDEF = 0.90
EN_IYI = "convnext_tiny_448px_lr0.0001_ls0.1_s42"
evaluate(df, EN_IYI, split="val", verbose=False)  # val tahminlerini üretir
print(save_threshold(EN_IYI, target=HEDEF))
confidence_band(load_predictions(EN_IYI, "test"), load_threshold(EN_IYI))

## Renk augmentation deneyi

Hipotez: model rengi/doluluğu fazla kullanıyor; ilkokul erkeklerin seyrek ve az renkli mutlu çizimlerini "üzgün" sanıyor.
`aug="renk"` (renk bozma + gri tonlama) ile aynı ayar 3 seed. Tek değişen augmentation. T4'te her biri ≈ 25 dk.

In [ ]:
for seed in [42, 1, 2]:
    train(df, "convnext_tiny", size=448, seed=seed, aug="renk", note="renk aug")

## Karşılaştırma (val)

Temel ve renk augmentation'lı modeller val'de, seed ortalamasıyla. Bakılacaklar: genel F1 düşmeden ilkokul erkek F1 artıyor mu,
mutlu→üzgün hatası azalıyor mu? Karar val'e göre verilir; test'e sadece seçilen model için bakılır.

In [ ]:
TEMEL = ["convnext_tiny_448px_lr0.0001_ls0.1_s42", "convnext_tiny_448_ep8_lr1e-4_ls0.1_s1", "convnext_tiny_448_ep8_lr1e-4_ls0.1_s2"]
RENK = [f"convnext_tiny_448_ep8_lr1e-4_ls0.1_s{s}_aug-renk" for s in [42, 1, 2]]
for d in TEMEL + RENK:
    if not os.path.exists(f"{PATHS['output']}/tahminler/{d}_val.csv"):
        evaluate(df, d, split="val", verbose=False)
tek, ozet = compare({**{d: "temel" for d in TEMEL}, **{d: "renk" for d in RENK}}, split="val")
display(tek, ozet)

## Tüm deneyler

In [ ]:
pd.read_csv(f"{PATHS['output']}/deneyler.csv").sort_values("en_iyi_val_f1", ascending=False)